# MDR-TS v20.6
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v20.5
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_8.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**
- Double pass to mimic classification. See notebook for more details

## 0. Imports

In [25]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor

import torch

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

imports loaded
using: cpu


In [2]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: False
environment setup complete


In [3]:
VERSION = "v20"
SUBVERSION = "v20.5"
RUN_NAME = "mdr_ts_v20_5"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v20/v20.5

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [4]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/train.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/val.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/test.csv


In [5]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 499

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'soil_moisture_5cm', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08', 'J_bio_bio09', 'J_bio_bio10']


In [6]:
TARGET_COL = "soil_moisture_5cm"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS_BASE = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_ampm_diff_interp",

    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "V_ema_G_API_kobs7",
    "V_ema_G_API_kobs14",
    "V_ema_G_API_kobs30",
    "V_rollmean_G_API_kobs7",
    "V_rollmean_G_API_kobs14",

    "A_d_E_SAR_diff_kobs14",

    "V_ema_LST_modis_kobs7",
    "A_d_LST_modis_kobs14",
    "V_rollmin_LST_modis_kobs30",

    "V_rollmean_s2_b11_kobs7",

    "year_frac", "sin_year", "cos_year",
    "API_x_year", "SMAP_x_year",

    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",
]

FEATURE_COLS_DRY = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_sm_interp_diff1",
    "A_d_SMAP_sm_interp_kobs14",

    "V_ema_LST_modis_kobs7",
    "V_rollmin_LST_modis_kobs30",
    "A_d_LST_modis_kobs14",

    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",

    "G_API",
    "V_ema_G_API_kobs14",
    "C_lag_G_API_kobs1",

    "V_rollmean_s2_b11_kobs7",

    "year_frac", "sin_year", "cos_year",
]

FEATURE_COLS_WET = [
    "SMAP_sm_interp_diff1",
    "SMAP_sm_interp_rollstd7",
    "SMAP_sm_interp_rollrange7",
    "SMAP_sm_interp_pctchg",
    "A_d_SMAP_sm_interp_kobs7",
    "A_grad_SMAP_sm_interp_kobs7",
    "A_pct_SMAP_sm_interp",

    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "V_rollstd_G_API_kobs7",
    "V_rollcv_G_API_kobs7",
    "A_d_G_API_kobs7",

    "A_d_E_SAR_diff_kobs1",
    "A_d_E_SAR_diff_kobs7",
    "A_grad_E_SAR_diff_kobs7",
    "A_grad_E_SAR_ratio_kobs7",
    "V_rollstd_E_SAR_diff_kobs7",
    "V_rollstd_E_SAR_ratio_kobs7",

    "V_rollstd_F_NDMI_kobs7",
    "A_d_F_NDMI_kobs7",

    "year_frac", "sin_year", "cos_year",

    "slope", "elev",
]

def _check_cols(df, cols, name):
    missing = sorted(set(cols) - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns in {name}: {missing}")

for _df_name, _df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _check_cols(_df, KEEP_META_COLS + [TARGET_COL], _df_name)
    _check_cols(_df, FEATURE_COLS_BASE, f"{_df_name} (BASE)")
    _check_cols(_df, FEATURE_COLS_DRY, f"{_df_name} (DRY)")
    _check_cols(_df, FEATURE_COLS_WET, f"{_df_name} (WET)")

print("Columns locked")
print("  BASE features:", len(FEATURE_COLS_BASE))
print("  DRY  features:", len(FEATURE_COLS_DRY))
print("  WET  features:", len(FEATURE_COLS_WET))
print("  Target:", TARGET_COL)

Columns locked
  BASE features: 28
  DRY  features: 21
  WET  features: 26
  Target: soil_moisture_5cm


In [7]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics, title=None, decimals=5):
    preferred_order = [
        "n", "r2", "mae", "rmse", "ubrmse", "bias",
        "med_ae", "p90_ae",
        "q05_err", "q50_err", "q95_err",
    ]

    keys = list(metrics.keys())
    if "_" in keys[0]:
        prefix = keys[0].split("_")[0] + "_"
    else:
        prefix = ""

    ordered_keys = []
    for k in preferred_order:
        full_key = prefix + k
        if full_key in metrics:
            ordered_keys.append(full_key)

    for k in metrics:
        if k not in ordered_keys:
            ordered_keys.append(k)

    if title:
        print(f"\n{title}")
    print("=" * 42)
    print(f"{'Metric':<18} {'Value':>18}")
    print("-" * 42)

    for k in ordered_keys:
        v = metrics[k]

        if isinstance(v, int):
            val_str = f"{v:,}"
        else:
            val_str = f"{v:.{decimals}f}"

        print(f"{k:<18} {val_str:>18}")

    print("=" * 42)

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [8]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     6868
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2017-01-01 -- 2020-12-31

VAL
  rows:     2720
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-01-01 -- 2022-12-31

TEST
  rows:     4016
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2023-01-01 -- 2025-12-31

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


In [9]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_trainval_base = trainval_df_d[FEATURE_COLS_BASE].copy()
y_trainval = trainval_df_d[TARGET_COL].copy()

X_test_base = test_df[FEATURE_COLS_BASE].copy()
y_test = test_df[TARGET_COL].copy()

X_trainval_dry = trainval_df_d[FEATURE_COLS_DRY].copy()
X_test_dry = test_df[FEATURE_COLS_DRY].copy()

X_trainval_wet = trainval_df_d[FEATURE_COLS_WET].copy()
X_test_wet = test_df[FEATURE_COLS_WET].copy()

print("\nMatrix shapes:")
print("  BASE trainval:", X_trainval_base.shape)
print("  DRY  trainval:", X_trainval_dry.shape)
print("  WET  trainval:", X_trainval_wet.shape)
print("  BASE test:    ", X_test_base.shape)
print("  DRY  test:    ", X_test_dry.shape)
print("  WET  test:    ", X_test_wet.shape)


Matrix shapes:
  BASE trainval: (9588, 28)
  DRY  trainval: (9588, 21)
  WET  trainval: (9588, 26)
  BASE test:     (4016, 28)
  DRY  test:     (4016, 21)
  WET  test:     (4016, 26)


### Inner Split

In [10]:
y_tv = np.asarray(y_trainval).ravel()

n = len(y_tv)
cut = int(0.8 * n)  # last 20% is inner-val

idx_tr = np.arange(0, cut)
idx_va = np.arange(cut, n)

print("Inner split sizes:")
print("  train:", len(idx_tr))
print("  val:  ", len(idx_va))

Inner split sizes:
  train: 7670
  val:   1918


In [11]:
t1 = 0.20
t2 = 0.313

reg_tv = np.zeros_like(y_tv, dtype=int)
reg_tv[(y_tv > t1) & (y_tv <= t2)] = 1
reg_tv[y_tv > t2] = 2

# masks per regime for inner train/val
masks = {}
for r in [0, 1, 2]:
    masks[r] = {
        "tr": (reg_tv[idx_tr] == r),
        "va": (reg_tv[idx_va] == r),
    }
    print(r, "inner-tr:", masks[r]["tr"].sum(), "inner-va:", masks[r]["va"].sum())

0 inner-tr: 3577 inner-va: 703
1 inner-tr: 3094 inner-va: 791
2 inner-tr: 999 inner-va: 424


### Base Model

In [12]:
XGB_PARAMS_DRY = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

XGB_PARAMS_TRANSITION = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    max_depth=7,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.85,
    n_estimators=8000,
    learning_rate=0.03,
    reg_lambda=3.0,
    reg_alpha=0.05,
)

XGB_PARAMS_WET = dict(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    max_depth=10,
    min_child_weight=1,
    subsample=1.0,
    colsample_bytree=0.9,
    n_estimators=6000,
    learning_rate=0.03,
    reg_lambda=0.3,
    reg_alpha=0.0,
)

In [13]:
xgb_dry = XGBRegressor(**XGB_PARAMS_DRY)
xgb_transition = XGBRegressor(**XGB_PARAMS_TRANSITION)
xgb_wet = XGBRegressor(**XGB_PARAMS_WET)

mask_dry = reg_tv == 0
mask_transition = reg_tv == 1
mask_wet = reg_tv == 2

In [14]:
xgb_dry.fit(X_trainval_base, y_trainval, verbose=0)

pred_dry_tv = xgb_dry.predict(X_trainval_base)   # length = len(trainval)
pred_dry_test = xgb_dry.predict(X_test_base)     # length = len(test)

In [15]:
X_tv_aug = np.column_stack([X_trainval_wet, pred_dry_tv])
X_test_aug = np.column_stack([X_test_wet, pred_dry_test])

xgb_transition.fit(
    X_tv_aug[mask_transition],
    y_trainval[mask_transition],
    verbose=0
)

pred_transition = xgb_transition.predict(X_test_aug)

In [16]:
xgb_wet.fit(
    X_tv_aug[mask_wet],
    y_trainval[mask_wet],
    verbose=0
)
pred_wet = xgb_wet.predict(X_test_aug)

In [17]:
y_test_arr = np.asarray(y_test).ravel()

reg_test = np.zeros_like(y_test_arr, dtype=int)
reg_test[(y_test_arr > t1) & (y_test_arr <= t2)] = 1
reg_test[y_test_arr > t2] = 2

mask_dry_test = reg_test == 0
mask_transition_test = reg_test == 1
mask_wet_test = reg_test == 2

print("Test regime counts:")
print("  dry:", mask_dry_test.sum())
print("  transition:", mask_transition_test.sum())
print("  wet:", mask_wet_test.sum())

Test regime counts:
  dry: 1509
  transition: 2015
  wet: 492


In [18]:
metrics_dashboard(
    y_test_arr[mask_dry_test],
    np.asarray(pred_dry_test).ravel()[mask_dry_test],
    name="DRY REGIME (Base/Anchor) - slice",
    return_dict=False
)

In [19]:
neat_print(get_metrics_dict(y_test_arr[mask_dry_test], np.asarray(pred_dry_test).ravel()[mask_dry_test], prefix=""), "DRY REGIME")


DRY REGIME
Metric                          Value
------------------------------------------
n                               1,509
r2                            0.33437
mae                           0.03288
rmse                          0.04525
ubrmse                        0.03919
bias                         -0.02262
med_ae                        0.02253
p90_ae                        0.07470
q05_err                      -0.09031
q50_err                      -0.01637
q95_err                       0.03918


In [20]:
metrics_dashboard(
    y_test_arr[mask_transition_test],
    np.asarray(pred_transition).ravel()[mask_transition_test],
    name="TRANSITION specialist - slice",
    return_dict=False
)

In [21]:
neat_print(get_metrics_dict(y_test_arr[mask_transition_test], np.asarray(pred_transition).ravel()[mask_transition_test], prefix=""), "TRANSITION REGIME")


TRANSITION REGIME
Metric                          Value
------------------------------------------
n                               2,015
r2                            0.14745
mae                           0.02331
rmse                          0.02997
ubrmse                        0.02990
bias                         -0.00204
med_ae                        0.01929
p90_ae                        0.04969
q05_err                      -0.05511
q50_err                      -0.00083
q95_err                       0.04632


In [22]:
metrics_dashboard(
    y_test_arr[mask_wet_test],
    np.asarray(pred_wet).ravel()[mask_wet_test],
    name="WET specialist - slice",
    return_dict=False
)

In [23]:
neat_print(get_metrics_dict(y_test_arr[mask_wet_test], np.asarray(pred_wet).ravel()[mask_wet_test], prefix=""), "WET REGIME")


WET REGIME
Metric                          Value
------------------------------------------
n                                 492
r2                           -0.31852
mae                           0.01181
rmse                          0.01659
ubrmse                        0.01612
bias                          0.00395
med_ae                        0.00807
p90_ae                        0.02969
q05_err                      -0.01742
q50_err                       0.00048
q95_err                       0.03788


In [24]:
neat_print(get_metrics_dict(y_test_arr[mask_dry_test], np.asarray(pred_dry_test).ravel()[mask_dry_test], prefix=""), "DRY REGIME")
neat_print(get_metrics_dict(y_test_arr[mask_transition_test], np.asarray(pred_transition).ravel()[mask_transition_test], prefix=""), "TRANSITION REGIME")
neat_print(get_metrics_dict(y_test_arr[mask_wet_test], np.asarray(pred_wet).ravel()[mask_wet_test], prefix=""), "WET REGIME")


DRY REGIME
Metric                          Value
------------------------------------------
n                               1,509
r2                            0.33437
mae                           0.03288
rmse                          0.04525
ubrmse                        0.03919
bias                         -0.02262
med_ae                        0.02253
p90_ae                        0.07470
q05_err                      -0.09031
q50_err                      -0.01637
q95_err                       0.03918

TRANSITION REGIME
Metric                          Value
------------------------------------------
n                               2,015
r2                            0.14745
mae                           0.02331
rmse                          0.02997
ubrmse                        0.02990
bias                         -0.00204
med_ae                        0.01929
p90_ae                        0.04969
q05_err                      -0.05511
q50_err                      -0.00083
q95_err  

---

## Self Gated Version

In [44]:
t2 = 0.313 * 0.9

In [45]:
def make_regime_labels_from_pred(pred, t1, t2):
    pred = np.asarray(pred).ravel()
    reg = np.zeros_like(pred, dtype=int)
    reg[(pred > t1) & (pred <= t2)] = 1
    reg[pred > t2] = 2
    return reg

def temporal_oof_predictions(model, X, y, n_splits=5):
    X = X.copy()
    y = np.asarray(y).ravel()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof_pred = np.full(len(X), np.nan, dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), start=1):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y[tr_idx], verbose=0)
        oof_pred[va_idx] = m.predict(X.iloc[va_idx])

        print(
            f"Fold {fold}: "
            f"train={len(tr_idx):,}  val={len(va_idx):,}  "
            f"filled={np.isfinite(oof_pred).sum():,}/{len(oof_pred):,}"
        )

    return oof_pred

In [46]:
XGB_PARAMS_ANCHOR = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

xgb_anchor_proto = XGBRegressor(**XGB_PARAMS_ANCHOR)

In [47]:
anchor_oof_tv = temporal_oof_predictions(
    xgb_anchor_proto,
    X_trainval_base,
    y_trainval,
    n_splits=5
)

valid_oof_mask = np.isfinite(anchor_oof_tv)
print("OOF coverage:", int(valid_oof_mask.sum()), "/", len(anchor_oof_tv))

reg_pseudo_tv = make_regime_labels_from_pred(anchor_oof_tv[valid_oof_mask], t1, t2)

print("Pseudo regime counts from anchor OOF predictions:")
print("  dry:", int((reg_pseudo_tv == 0).sum()))
print("  transition:", int((reg_pseudo_tv == 1).sum()))
print("  wet:", int((reg_pseudo_tv == 2).sum()))

Fold 1: train=1,598  val=1,598  filled=1,598/9,588
Fold 2: train=3,196  val=1,598  filled=3,196/9,588
Fold 3: train=4,794  val=1,598  filled=4,794/9,588
Fold 4: train=6,392  val=1,598  filled=6,392/9,588
Fold 5: train=7,990  val=1,598  filled=7,990/9,588
OOF coverage: 7990 / 9588
Pseudo regime counts from anchor OOF predictions:
  dry: 3533
  transition: 2890
  wet: 1567


In [48]:
X_tv_wet_valid = X_trainval_wet.iloc[valid_oof_mask].copy()
y_tv_valid = np.asarray(y_trainval).ravel()[valid_oof_mask]

X_tv_aug_self = np.column_stack([
    X_tv_wet_valid.to_numpy(),
    anchor_oof_tv[valid_oof_mask]
])

mask_dry_pseudo = reg_pseudo_tv == 0
mask_transition_pseudo = reg_pseudo_tv == 1
mask_wet_pseudo = reg_pseudo_tv == 2

print("Pseudo-routed train sizes:")
print("  dry:", int(mask_dry_pseudo.sum()))
print("  transition:", int(mask_transition_pseudo.sum()))
print("  wet:", int(mask_wet_pseudo.sum()))

Pseudo-routed train sizes:
  dry: 3533
  transition: 2890
  wet: 1567


In [49]:
xgb_transition_self = XGBRegressor(**XGB_PARAMS_TRANSITION)
xgb_wet_self = XGBRegressor(**XGB_PARAMS_WET)

if mask_transition_pseudo.sum() > 0:
    xgb_transition_self.fit(
        X_tv_aug_self[mask_transition_pseudo],
        y_tv_valid[mask_transition_pseudo],
        verbose=0
    )

if mask_wet_pseudo.sum() > 0:
    xgb_wet_self.fit(
        X_tv_aug_self[mask_wet_pseudo],
        y_tv_valid[mask_wet_pseudo],
        verbose=0
    )

print("self-gated specialists trained")

self-gated specialists trained


In [50]:
xgb_anchor_final = XGBRegressor(**XGB_PARAMS_ANCHOR)
xgb_anchor_final.fit(X_trainval_base, y_trainval, verbose=0)

anchor_pred_test = xgb_anchor_final.predict(X_test_base)
X_test_aug_self = np.column_stack([
    X_test_wet.to_numpy(),
    anchor_pred_test
])

pred_transition_self = xgb_transition_self.predict(X_test_aug_self)
pred_wet_self = xgb_wet_self.predict(X_test_aug_self)

reg_test_hat = make_regime_labels_from_pred(anchor_pred_test, t1, t2)

mask_dry_hat = reg_test_hat == 0
mask_transition_hat = reg_test_hat == 1
mask_wet_hat = reg_test_hat == 2

print("test pseudo-gate counts:")
print("  dry:", int(mask_dry_hat.sum()))
print("  transition:", int(mask_transition_hat.sum()))
print("  wet:", int(mask_wet_hat.sum()))

test pseudo-gate counts:
  dry: 1540
  transition: 1408
  wet: 1068


In [51]:
pred_final_self = np.zeros_like(anchor_pred_test, dtype=float)

# dry branch uses anchor directly
pred_final_self[mask_dry_hat] = anchor_pred_test[mask_dry_hat]

# transition specialist
pred_final_self[mask_transition_hat] = pred_transition_self[mask_transition_hat]

# wet specialist
pred_final_self[mask_wet_hat] = pred_wet_self[mask_wet_hat]

metrics_dashboard(
    y_test,
    pred_final_self,
    name="FINAL | Double Pass Self-Gated",
    return_dict=False
)

In [52]:
neat_print(
    get_metrics_dict(np.asarray(y_test).ravel(), pred_final_self, prefix=""),
    "DOUBLE PASS SELF-GATED"
)


DOUBLE PASS SELF-GATED
Metric                          Value
------------------------------------------
n                               4,016
r2                            0.75991
mae                           0.03510
rmse                          0.04614
ubrmse                        0.04507
bias                         -0.00986
med_ae                        0.02733
p90_ae                        0.07472
q05_err                      -0.08570
q50_err                      -0.00952
q95_err                       0.05928


---

## Resiudal Double Pass Version

In [53]:
X_tv_aug = np.column_stack([X_trainval_wet, pred_dry_tv])
X_test_aug = np.column_stack([X_test_wet, pred_dry_test])

y_trainval_arr = np.asarray(y_trainval).ravel()
y_test_arr = np.asarray(y_test).ravel()

resid_tv = y_trainval_arr - np.asarray(pred_dry_tv).ravel()

print("Augmented trainval shape:", X_tv_aug.shape)
print("Augmented test shape:    ", X_test_aug.shape)
print("Residual target shape:   ", resid_tv.shape)

Augmented trainval shape: (9588, 27)
Augmented test shape:     (4016, 27)
Residual target shape:    (9588,)


In [54]:
base_pred_tv = np.asarray(pred_dry_tv).ravel()
base_pred_test = np.asarray(pred_dry_test).ravel()

mask_dry_hat_tv = base_pred_tv <= t1
mask_transition_hat_tv = (base_pred_tv > t1) & (base_pred_tv <= t2)
mask_wet_hat_tv = base_pred_tv > t2

mask_dry_hat_test = base_pred_test <= t1
mask_transition_hat_test = (base_pred_test > t1) & (base_pred_test <= t2)
mask_wet_hat_test = base_pred_test > t2

print("Pseudo-gated TRAINVAL counts:")
print("  dry:       ", int(mask_dry_hat_tv.sum()))
print("  transition:", int(mask_transition_hat_tv.sum()))
print("  wet:       ", int(mask_wet_hat_tv.sum()))

print("\nPseudo-gated TEST counts:")
print("  dry:       ", int(mask_dry_hat_test.sum()))
print("  transition:", int(mask_transition_hat_test.sum()))
print("  wet:       ", int(mask_wet_hat_test.sum()))

Pseudo-gated TRAINVAL counts:
  dry:        4252
  transition: 2627
  wet:        2709

Pseudo-gated TEST counts:
  dry:        1540
  transition: 1408
  wet:        1068


In [55]:
xgb_transition_resid = XGBRegressor(**XGB_PARAMS_TRANSITION)
xgb_wet_resid = XGBRegressor(**XGB_PARAMS_WET)

if mask_transition_hat_tv.sum() > 0:
    xgb_transition_resid.fit(
        X_tv_aug[mask_transition_hat_tv],
        resid_tv[mask_transition_hat_tv],
        verbose=0
    )

wet_bias = resid_tv[mask_wet_hat_tv].mean()

if mask_wet_hat_tv.sum() > 0:
    xgb_wet_resid.fit(
        X_tv_aug[mask_wet_hat_tv],
        resid_tv[mask_wet_hat_tv] - wet_bias,
        verbose=0
    )

print("residual specialists trained.")

residual specialists trained.


In [56]:
pred_transition_resid_test = np.zeros_like(base_pred_test, dtype=float)
pred_wet_resid_test = np.zeros_like(base_pred_test, dtype=float)

if mask_transition_hat_test.sum() > 0:
    pred_transition_resid_test[mask_transition_hat_test] = xgb_transition_resid.predict(
        X_test_aug[mask_transition_hat_test]
    )

if mask_wet_hat_test.sum() > 0:
    pred_wet_resid_test[mask_wet_hat_test] = xgb_wet_resid.predict(
        X_test_aug[mask_wet_hat_test]
    )

pred_wet_resid_test = np.clip(pred_wet_resid_test, -0.05, 0.05)

print("residual corrections predicted.")

residual corrections predicted.


In [57]:
pred_final_resid = np.asarray(base_pred_test).copy()

# transition: anchor + transition correction
pred_final_resid[mask_transition_hat_test] = (
    base_pred_test[mask_transition_hat_test]
    + pred_transition_resid_test[mask_transition_hat_test]
)

# wet: anchor + wet correction
pred_final_resid[mask_wet_hat_test] = (
    base_pred_test[mask_wet_hat_test]
    + pred_wet_resid_test[mask_wet_hat_test]
)

metrics_dashboard(
    y_test_arr,
    pred_final_resid,
    name="FINAL | Residual Double Pass Self-Gated",
    return_dict=False
)

neat_print(
    get_metrics_dict(y_test_arr, pred_final_resid, prefix=""),
    "RESIDUAL DOUBLE PASS SELF-GATED"
)


RESIDUAL DOUBLE PASS SELF-GATED
Metric                          Value
------------------------------------------
n                               4,016
r2                            0.80129
mae                           0.03193
rmse                          0.04198
ubrmse                        0.04185
bias                         -0.00320
med_ae                        0.02458
p90_ae                        0.06851
q05_err                      -0.07332
q50_err                      -0.00500
q95_err                       0.06245


In [58]:
metrics_dashboard(
    y_test_arr,
    base_pred_test,
    name="BASE | Anchor Only",
    return_dict=False
)

neat_print(
    get_metrics_dict(y_test_arr, base_pred_test, prefix=""),
    "ANCHOR ONLY"
)


ANCHOR ONLY
Metric                          Value
------------------------------------------
n                               4,016
r2                            0.80199
mae                           0.03186
rmse                          0.04190
ubrmse                        0.04178
bias                         -0.00324
med_ae                        0.02458
p90_ae                        0.06844
q05_err                      -0.07353
q50_err                      -0.00503
q95_err                       0.06218


In [59]:
slices_resid = {
    "Dry": (
        y_test_arr[mask_dry_hat_test],
        pred_final_resid[mask_dry_hat_test]
    ),
    "Transition": (
        y_test_arr[mask_transition_hat_test],
        pred_final_resid[mask_transition_hat_test]
    ),
    "Wet": (
        y_test_arr[mask_wet_hat_test],
        pred_final_resid[mask_wet_hat_test]
    ),
}

met_rows_resid = {}
for name, (yt, yp) in slices_resid.items():
    m = get_metrics_dict(yt, yp, prefix="")
    met_rows_resid[name] = {k: m[k] for k in ["n", "r2", "rmse", "ubrmse", "bias"]}
    print(
        f"{name:12s}: R²={m['r2']:+.4f}  RMSE={m['rmse']:.4f}  "
        f"ubRMSE={m['ubrmse']:.4f}  Bias={m['bias']:+.4f}"
    )

met_df_resid = pd.DataFrame(met_rows_resid).T
met_df_resid.index.name = "Predicted Gate"
met_df_resid.columns = ["n", "R²", "RMSE", "ubRMSE", "Bias"]
met_df_resid["n"] = met_df_resid["n"].astype(int)

display(
    met_df_resid.style
    .format({"n": "{:,}", "R²": "{:.4f}", "RMSE": "{:.4f}", "ubRMSE": "{:.4f}", "Bias": "{:+.4f}"})
    .background_gradient(subset=["R²"], cmap="Greens")
    .background_gradient(subset=["RMSE", "ubRMSE"], cmap="Reds_r")
    .set_caption("Test-set regime metrics | residual double-pass self-gated")
)

Dry         : R²=+0.7017  RMSE=0.0378  ubRMSE=0.0378  Bias=-0.0014
Transition  : R²=+0.2438  RMSE=0.0455  ubRMSE=0.0453  Bias=+0.0033
Wet         : R²=-0.1210  RMSE=0.0429  ubRMSE=0.0405  Bias=-0.0143


,n,R²,RMSE,ubRMSE,Bias
Predicted Gate,,,,,
Dry,"1,540",0.7017,0.0378,0.0378,-0.0014
Transition,"1,408",0.2438,0.0455,0.0453,+0.0033
Wet,"1,068",-0.1210,0.0429,0.0405,-0.0143
